# 04b — ETA lúc nhận đơn và đối chiếu tính tay

Protocol B dự đoán tại lúc shipper nhận đơn. Mỗi mẫu ứng với một `order_id`,
`query_ts = accept_ts` và nhãn là số phút từ nhận đến giao. Ví dụ dùng **B + AOI + GPS**
trong benchmark validation với seed 42.

### 1. Mở đúng file đầu vào

`data/delivery_yt.csv` nằm trong repo. Trong Docker, file được mount chỉ đọc ở
`/home/jovyan/data/delivery_yt.csv`. Cell kiểm tra SHA-256 theo `config/prereg.yaml`.
Prefect đọc B từ Bronze tạo từ CSV. Phần tính tay đọc trực tiếp CSV và chạy
bằng kernel Python đã cài `requirements.txt`.

In [1]:
import hashlib
from math import asin, cos, fsum, isclose, radians, sin, sqrt
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import yaml

cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, cwd.parent, Path("/home/jovyan/work"))
             if (p / "config" / "prereg.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Mở notebook từ thư mục gốc dự án, thư mục notebooks hoặc Jupyter Docker.")
sys.path.insert(0, str(ROOT))

from scripts.run_eta_accept_benchmark import (
    TARGET, VARIANTS, fill_from_train_medians, validation_metrics, validation_rows,
)
from src.eta.dataset import SOURCE_COLUMNS, build_eta_samples, read_lade_csv
from src.eta.model import fit_lightgbm

prereg = yaml.safe_load((ROOT / "config" / "prereg.yaml").read_text(encoding="utf-8"))
source_config = prereg["data"]
source = ROOT / source_config["source"]
if not source.is_file():
    source = ROOT.parent / source_config["source"]

with source.open("rb") as handle:
    checksum = hashlib.file_digest(handle, "sha256").hexdigest()
assert checksum == source_config["sha256"], f"SHA-256 của {source} sai: {checksum}"

events = read_lade_csv(source)
orders = events.set_index("order_id", verify_integrity=True)
print(f"File cấu hình  {source_config['source']}")
print(f"SHA-256        {checksum}")
print(f"CSV            {len(events):,} dòng × {len(SOURCE_COLUMNS)} cột")
print(events[["order_id", "courier_id", "aoi_id", "accept_time", "delivery_time"]]
      .head(3).to_string(index=False))

File cấu hình  data/delivery_yt.csv
SHA-256        aa5bca7fdaa8c89d4ab50a2ed2b94fb09f608acc49fc1108a1a612e9c0f5a7ad
CSV            206,431 dòng × 17 cột
order_id  courier_id  aoi_id    accept_time  delivery_time
 2071340        3135     175 06-18 08:00:00 06-18 16:21:00
 3767291        3135     175 08-14 08:30:00 08-14 15:01:00
 2184812        3135     175 09-01 09:08:00 09-01 15:15:00


### 2. Kiểm tra thời gian, dựng mẫu và chia theo ngày

`read_lade_csv()` kiểm tra đúng 17 cột, gắn năm 2022 và tính thời lượng.
Builder giữ đơn có timestamp nhận, giao hợp lệ và thời lượng dương. Hai ngưỡng
24 giờ và 50 km áp dụng cho Spark LR/GBT ở notebook 04.

`build_eta_samples()` dựng đặc trưng tại lúc nhận và lấy tối đa 12 đơn cho mỗi
shipper mỗi ngày bằng seed lấy mẫu 42. Khi fit, chỉ dùng nhãn đã xuất hiện
trước ngày bắt đầu validation.

In [2]:
valid_time = events.accept_ts.notna() & events.delivery_ts.notna()
positive_duration = valid_time & events.delivery_minutes.gt(0)
samples = build_eta_samples(
    events, protocol="B", max_per_courier_day=source_config["max_queries_per_courier_day"],
    seed=source_config["query_seed"], include_aoi_gps=True,
)
train, validation = validation_rows(samples, orders)
first_validation_day = validation.query_ts.min().date()

counts = pd.Series({
    "Dòng gốc": len(events),
    "Timestamp hợp lệ và thời lượng dương": int(positive_duration.sum()),
    "Mẫu sau khi chọn tối đa 12 đơn mỗi shipper/ngày": len(samples),
    "Train còn nhãn quan sát trước validation": len(train),
    "Validation": len(validation),
    "Test giữ lại": int(samples.split.eq("test").sum()),
})
print(counts.to_string())
print(f"Validation bắt đầu {first_validation_day}")

Dòng gốc                                           206431
Timestamp hợp lệ và thời lượng dương               206062
Mẫu sau khi chọn tối đa 12 đơn mỗi shipper/ngày    123117
Train còn nhãn quan sát trước validation            73107
Validation                                          24647
Test giữ lại                                        25345
Validation bắt đầu 2022-08-19


### 3. Tính nhãn và đặc trưng từ một đơn thật

Chọn đơn validation sớm nhất có GPS hợp lệ và có từ một đến ba đơn đã giao
cùng AOI. Nhờ vậy có thể đọc từng dòng lịch sử dùng trong phép tính.

- Nhãn bằng giờ giao trừ giờ nhận.
- `aoi_past_count` đếm đơn cùng AOI đã giao tới mốc nhận. `aoi_past_avg_min` lấy
  tổng thời lượng của các đơn đó chia cho số đơn.
- `accept_to_stop_km` dùng Haversine từ GPS nhận đến tọa độ đích. Timestamp GPS
  phải tới lúc nhận.

LightGBM nhận đặc trưng ở lúc nhận. Giờ giao thực tế của đơn minh họa dùng để chấm sai số.

In [3]:
candidates = validation.loc[validation.aoi_past_count.between(1, 3)
                            & validation.accept_to_stop_km.notna()]
example = candidates.sort_values(["query_ts", "order_id"]).iloc[0]
order = orders.loc[example.order_id]
accepted = pd.Timestamp(f"2022-{order.accept_time}")
delivered = pd.Timestamp(f"2022-{order.delivery_time}")
hand_label = (delivered - accepted).total_seconds() / 60

history = events.loc[events.aoi_id.eq(order.aoi_id) & events.delivery_ts.le(accepted)
                     & events.delivery_ts.gt(events.accept_ts)].sort_values("delivery_ts")
hand_durations = (history.delivery_ts - history.accept_ts).dt.total_seconds() / 60
hand_count = len(history)
hand_mean = hand_durations.sum() / hand_count

start_lat, start_lng, stop_lat, stop_lng = map(
    radians, (order.accept_gps_lat, order.accept_gps_lng, order.lat, order.lng),
)
a = sin((stop_lat - start_lat) / 2)**2 + cos(start_lat) * cos(stop_lat) * sin((stop_lng - start_lng) / 2)**2
hand_distance = 2 * 6371.0088 * asin(sqrt(a))
feature_check = pd.DataFrame(
    {"Tính tay": [hand_label, hand_count, hand_mean, hand_distance],
     "Builder": [example[TARGET], example.aoi_past_count,
                 example.aoi_past_avg_min, example.accept_to_stop_km]},
    index=[TARGET, "aoi_past_count", "aoi_past_avg_min", "accept_to_stop_km"],
)
np.testing.assert_allclose(feature_check["Tính tay"], feature_check["Builder"], rtol=0, atol=1e-9)

print(f"Đơn {example.order_id}, AOI {order.aoi_id}")
print(f"Nhãn = {order.delivery_time} - {order.accept_time} = {hand_label:.0f} phút")
print(history[["order_id", "accept_time", "delivery_time"]]
      .assign(duration_min=hand_durations).to_string(index=False))
terms = " + ".join(f"{duration:g}" for duration in hand_durations)
print(f"Trung bình AOI = ({terms}) / {hand_count} = {hand_mean:.2f} phút")
print(f"GPS lúc nhận {order.accept_gps_time}  ({order.accept_gps_lat}, {order.accept_gps_lng})")
print(f"Tọa độ đích ({order.lat}, {order.lng})")
print(feature_check.to_string(float_format=lambda value: f"{value:.8f}"))

Đơn 2641862, AOI 18619
Nhãn = 08-19 11:26:00 - 08-19 08:06:00 = 200 phút
order_id    accept_time  delivery_time  duration_min
 1935945 07-13 08:55:00 07-13 11:44:00         169.0
Trung bình AOI = (169) / 1 = 169.00 phút
GPS lúc nhận 08-19 08:06:00  (37.15448, 119.94911)
Tọa độ đích (37.19522, 119.93456)
                             Tính tay      Builder
target_remaining_minutes 200.00000000 200.00000000
aoi_past_count             1.00000000   1.00000000
aoi_past_avg_min         169.00000000 169.00000000
accept_to_stop_km          4.70994039   4.70994039


### 4. Đưa đúng vector đặc trưng vào LightGBM

Điền thiếu bằng median trên train theo script benchmark. Model dùng validation
để dừng sớm. Hàng in bên dưới được đưa vào `predict()` cho đơn đang kiểm tra.

In [4]:
features = VARIANTS["protocol_b_aoi_gps"]
train_inputs, validation_inputs = fill_from_train_medians(train, validation, features)
model_input = validation_inputs.loc[[example.name], features]
print(model_input.T.rename(columns={example.name: "Giá trị đưa vào mô hình"}).to_string())

seed = prereg["model"]["seeds"][0]
model = fit_lightgbm(train_inputs, validation_inputs, features, seed=seed)
print(f"Seed {seed}, {model.best_iteration_} cây ở lần dừng tốt nhất.")

                             Giá trị đưa vào mô hình
pending_order_count                                0
completed_order_count                            325
courier_avg_min                           305.824615
distance_to_last_delivery_km                2.573285
candidate_lat                               37.19522
candidate_lng                              119.93456
candidate_aoi_type                                14
candidate_region_id                               95
accept_hour                                      8.1
aoi_past_avg_min                               169.0
aoi_past_count                                     1
accept_to_stop_km                            4.70994


Seed 42, 1963 cây ở lần dừng tốt nhất.


### 5. Lần theo các cây rồi cộng để kiểm tra dự đoán

Mỗi nút so một đặc trưng với ngưỡng. Đi trái nếu giá trị nhỏ hơn hoặc bằng
ngưỡng. Đi phải nếu lớn hơn. Khi tới lá, lấy `leaf_value` đã lưu trong cây.

Cộng ba giá trị lá đầu để so với `predict(num_iteration=3)` của cùng model.
Sau đó lặp phép cộng cho toàn bộ cây và chặn dưới ở 0 phút theo benchmark.

In [5]:
def follow_tree(node, row, record_path=False):
    path = []
    while "split_index" in node:
        assert node["decision_type"] == "<=" and node["missing_type"] != "Zero"
        column = features[node["split_feature"]]
        value = float(row[column])
        threshold = float(node["threshold"])
        child = "left_child" if value <= threshold else "right_child"
        if record_path:
            branch = "trái" if child == "left_child" else "phải"
            path.append(f"{column} ({value!r}) <= {threshold!r} → {branch}")
        node = node[child]
    return float(node["leaf_value"]), path

dump = model.booster_.dump_model(num_iteration=model.best_iteration_)
assert dump["num_tree_per_iteration"] == 1 and not dump["average_output"]
assert model_input.notna().all().all()
row = model_input.iloc[0]
leaves = []
for index, tree in enumerate(dump["tree_info"]):
    value, path = follow_tree(tree["tree_structure"], row, record_path=index < 3)
    leaves.append(value)
    if index < 3:
        print(f"Cây {index + 1}")
        print("\n".join(f"  {decision}" for decision in path))
        print(f"  Lá {value:.10f}")

hand_three = fsum(leaves[:3])
machine_three = float(model.predict(model_input, num_iteration=3)[0])
hand_full = max(0.0, fsum(leaves))
machine_full = max(0.0, float(model.predict(model_input)[0]))
assert isclose(hand_three, machine_three, rel_tol=0, abs_tol=1e-9)
assert isclose(hand_full, machine_full, rel_tol=0, abs_tol=1e-9)

summands = " + ".join(f"({value:.10f})" for value in leaves[:3])
print(f"Ba cây = {summands} = {hand_three:.10f} phút")
print(f"Máy dùng ba cây      {machine_three:.10f} phút")
print(f"Cộng đủ {len(leaves)} cây    {hand_full:.10f} phút")
print(f"Máy dùng đủ cây       {machine_full:.10f} phút")
print(f"Nhãn thực tế {hand_label:.2f} phút, sai số tuyệt đối {abs(hand_full - hand_label):.2f} phút")

Cây 1
  courier_avg_min (305.8246153846154) <= 206.9991007194245 → phải
  accept_to_stop_km (4.709940386827142) <= 3.9797023361747548 → phải
  pending_order_count (0.0) <= 1.5000000000000002 → trái
  courier_avg_min (305.8246153846154) <= 408.25178571428575 → trái
  accept_hour (8.1) <= 16.225000000000005 → trái
  Lá 200.7800000000
Cây 2
  aoi_past_avg_min (169.0) <= 210.15587044534416 → trái
  aoi_past_avg_min (169.0) <= 163.80672268907566 → phải
  courier_avg_min (305.8246153846154) <= 168.6915680473373 → phải
  accept_hour (8.1) <= 14.091666666666669 → trái
  aoi_past_avg_min (169.0) <= 178.3592943201377 → trái
  Lá -1.1454000000
Cây 3
  aoi_past_avg_min (169.0) <= 213.449494949495 → trái
  aoi_past_avg_min (169.0) <= 162.68383097165994 → phải
  courier_avg_min (305.8246153846154) <= 152.18176291793318 → phải
  aoi_past_avg_min (169.0) <= 178.3592943201377 → trái
  Lá -1.2235380000
Ba cây = (200.7800000000) + (-1.1454000000) + (-1.2235380000) = 198.4110620000 phút
Máy dùng ba cây   

### 6. Chấm cả tập validation

Đối chiếu một đơn kiểm tra cách mô hình tính ra dự đoán. MAE đo trung bình sai số
tuyệt đối của mọi đơn validation. ACC@30 là tỷ lệ đơn có sai số tối đa 30 phút.
Tập test dành cho lần chấm sau khi chốt cấu hình.

In [6]:
validation_prediction = model.predict(validation_inputs[features])
metrics = validation_metrics(validation_inputs[TARGET].to_numpy(), validation_prediction)
print(f"Validation, seed {seed}, {len(validation_inputs):,} đơn")
print(pd.Series(metrics).to_string(float_format=lambda value: f"{value:.8f}"))

Validation, seed 42, 24,647 đơn
acc20_pct            22.61938573
acc30_pct            32.45019678
mae_min             109.45113225
p90_abs_error_min   222.75541744


### Chạy bước B đăng ký để ghi bảng

Khi cần công bố kết quả, chạy `pipeline/flow.py`. Cell tùy chọn dưới đây gọi
bước B đăng ký trong Docker với Bronze và MySQL sẵn sàng. Phần tính tay
ở trên chạy riêng và chỉ đọc CSV.

In [7]:
RUN_REGISTERED_B = False
if RUN_REGISTERED_B:
    pipeline_dir = ROOT / "pipeline"
    if not pipeline_dir.is_dir():
        pipeline_dir = ROOT.parent / "pipeline"
    sys.path.insert(0, str(pipeline_dir))

    import steps

    spark = steps.get_spark("Delivery - Protocol B")
    print(steps.build_eta_safe(spark))

<details>
<summary>Kết quả B đăng ký đã lưu từ lần chạy trước</summary>

Lần chạy dưới đây chấm Protocol B đăng ký trên test. Phần thực hành ở trên dùng biến thể B + AOI + GPS trên validation.

~~~text
{'run_id': 'ee33b1bcc0864b3db7fd8937600d8aa0',
 'train_rows': 73125,
 'validation_rows': 24647,
 'test_rows': 25345,
 'test_mae': 117.45598968125002,
 'test_rmse': 268.02749142476677,
 'seeds': [42, 43, 44]}
~~~

</details>